# 📖 Notebook 1: Photo Upload & Storage Pipeline

Instagram handles **100 million photo/video uploads per day**. That's ~1,150 uploads per second.  
Sending all that media through your app server would crush it — so Instagram doesn't.

Instead, the app server generates a **pre-signed URL**, and the client uploads directly to object storage (S3).  
This notebook shows you exactly how that works.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why we store photos in object storage (S3/MinIO) instead of a database
- How **pre-signed URLs** let clients upload directly to storage
- The full upload pipeline: create post → get URL → upload → confirm
- How to generate **thumbnails** (multiple image sizes for different devices)
- How a **CDN** would serve these images to users worldwide

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/instagram
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `instagram_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`
- **MinIO Console** (Object Storage GUI): http://localhost:9001  
  Login: `minioadmin` / `minioadmin`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import boto3
from botocore.client import Config
from PIL import Image
import io
import time
import json

# ── Database connection ──────────────────────────────────
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "instagram_demo",
    "user": "demo",
    "password": "demo"
}

# ── MinIO (S3-compatible) connection ─────────────────────
MINIO_CONFIG = {
    "endpoint_url": "http://localhost:9000",
    "aws_access_key_id": "minioadmin",
    "aws_secret_access_key": "minioadmin",
}
BUCKET_NAME = "instagram-photos"

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_s3():
    """Create an S3 client that talks to our local MinIO."""
    return boto3.client(
        "s3",
        endpoint_url=MINIO_CONFIG["endpoint_url"],
        aws_access_key_id=MINIO_CONFIG["aws_access_key_id"],
        aws_secret_access_key=MINIO_CONFIG["aws_secret_access_key"],
        config=Config(signature_version="s3v4"),
        region_name="us-east-1"
    )

# ── Test connections ─────────────────────────────────────
try:
    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM users")
    print(f"✅ PostgreSQL connected — {cur.fetchone()[0]} users")
    cur.execute("SELECT COUNT(*) FROM posts")
    print(f"   {cur.fetchone()[0]} posts in database")
    conn.close()
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    s3 = get_s3()
    # Create our bucket if it doesn't exist
    existing = [b["Name"] for b in s3.list_buckets()["Buckets"]]
    if BUCKET_NAME not in existing:
        s3.create_bucket(Bucket=BUCKET_NAME)
        print(f"✅ MinIO connected — created bucket '{BUCKET_NAME}'")
    else:
        print(f"✅ MinIO connected — bucket '{BUCKET_NAME}' exists")
except Exception as e:
    print(f"❌ MinIO failed: {e}")
    print("   Run: docker compose up -d")

## 🤔 Why Not Store Photos in the Database?

Beginners often ask: "Why not just put the image bytes in a PostgreSQL column?"

Here's why:

| Approach | Max File Size | Cost per GB/month | Read Speed | Scalability |
|----------|--------------|-------------------|------------|-------------|
| PostgreSQL BYTEA column | ~1 GB (practical) | ~$0.10–$0.25 (EBS) | Slow (disk I/O + parsing) | Vertical only |
| S3 / MinIO | 5 TB per object | ~$0.023 | Fast (HTTP streaming) | Infinite horizontal |

**The key insight**: databases are optimised for *structured queries* (WHERE, JOIN, INDEX).  
Object stores are optimised for *storing and serving large binary files* (photos, videos).

Instagram stores **post metadata** (caption, author, timestamps) in the database,  
and the **actual photo bytes** in S3. The database row just holds a `media_key` that points to S3.

```
┌──────────── PostgreSQL ────────────┐     ┌──────── S3/MinIO ────────┐
│ posts table                        │     │                          │
│ id=42, author=7, caption="Sunset" │     │ photos/user_7/post_42.jpg│
│ media_key="photos/user_7/post_42" ─┼────►│ (actual JPEG bytes)      │
└────────────────────────────────────┘     └──────────────────────────┘
```

## 📤 The Upload Pipeline

When you tap "Share" on Instagram, here's what happens behind the scenes:

```
Step 1: Client asks server "I want to upload a photo"
        POST /posts → {postId, presignedUrl}

Step 2: Client uploads photo DIRECTLY to S3 using the pre-signed URL
        PUT <presignedUrl> → (photo bytes go straight to S3)

Step 3: Client tells server "upload is done"
        PATCH /posts/{postId} → {status: 'complete'}
```

**Why pre-signed URLs?** They let the client upload directly to S3 without  
the photo bytes ever touching your app server. This saves bandwidth, CPU, and money.

A pre-signed URL is a normal S3 URL with a **temporary signature** attached.  
It says: "anyone with this URL can upload to this exact location, but only for the next 60 minutes."

### Step 1: Create Post Metadata & Get Pre-signed URL

The server creates a database row for the post (with status `pending`)  
and generates a pre-signed URL that allows uploading to a specific S3 path.

In [ ]:
def create_post(author_id: int, caption: str) -> dict:
    """
    Step 1 of the upload pipeline.
    Creates post metadata in the database and returns a pre-signed URL
    for the client to upload the photo directly to MinIO/S3.
    """
    conn = get_db()
    cur = conn.cursor()
    s3 = get_s3()

    # Decide where this photo will live in S3
    # We use a predictable path: photos/user_{id}/post_{post_id}.jpg
    # But we don't have the post_id yet — so we insert first, then update.
    cur.execute(
        """INSERT INTO posts (author_id, caption, media_key, media_upload_status)
           VALUES (%s, %s, 'pending', 'pending')
           RETURNING id""",
        (author_id, caption)
    )
    post_id = cur.fetchone()[0]

    # Now we know the post_id — build the S3 key
    media_key = f"photos/user_{author_id}/post_{post_id}.jpg"
    cur.execute(
        "UPDATE posts SET media_key = %s WHERE id = %s",
        (media_key, post_id)
    )
    conn.commit()

    # Generate a pre-signed URL (valid for 1 hour)
    presigned_url = s3.generate_presigned_url(
        "put_object",
        Params={"Bucket": BUCKET_NAME, "Key": media_key},
        ExpiresIn=3600  # 1 hour
    )

    conn.close()
    return {
        "post_id": post_id,
        "media_key": media_key,
        "presigned_url": presigned_url,
        "status": "pending"
    }

# Let's try it! User 1 wants to upload a sunset photo.
result = create_post(author_id=1, caption="Beautiful sunset at the beach! 🌅")
print(f"📝 Created post #{result['post_id']}")
print(f"   Media key: {result['media_key']}")
print(f"   Status: {result['status']}")
print(f"\n🔗 Pre-signed URL (first 100 chars):")
print(f"   {result['presigned_url'][:100]}...")

### Step 2: Upload Photo Directly to Object Storage

In a real app, the mobile client would use the pre-signed URL to upload.  
Here we'll simulate it by creating a small test image and uploading it.

Notice the photo bytes **never touch our Python app server** — they go straight to MinIO.

In [ ]:
import requests

def create_test_image(width=800, height=600, color="orange"):
    """Create a simple test image in memory (no file needed)."""
    img = Image.new("RGB", (width, height), color)
    buffer = io.BytesIO()
    img.save(buffer, format="JPEG")
    buffer.seek(0)
    return buffer

# Create a fake "sunset photo"
photo_bytes = create_test_image(800, 600, "orange")
print(f"📷 Created test image: {len(photo_bytes.getvalue())} bytes")

# Upload using the pre-signed URL — this goes directly to MinIO!
response = requests.put(
    result["presigned_url"],
    data=photo_bytes.getvalue(),
    headers={"Content-Type": "image/jpeg"}
)

if response.status_code == 200:
    print(f"✅ Photo uploaded directly to MinIO!")
    print(f"   Location: {BUCKET_NAME}/{result['media_key']}")
    print(f"   → Open MinIO Console (http://localhost:9001) to see it")
else:
    print(f"❌ Upload failed: {response.status_code} {response.text}")

### Step 3: Confirm Upload & Mark Post as Complete

After the client finishes uploading, it tells the server "I'm done."  
The server updates the post status from `pending` to `complete`.

Only posts with status `complete` should appear in feeds.

In [ ]:
def confirm_upload(post_id: int) -> dict:
    """
    Step 3 of the upload pipeline.
    Marks the post as complete after the client confirms the upload.
    The storage check is the important part: a client that crashed mid-upload
    will still send the confirmation, and we must not mark that post live.
    """
    conn = get_db()
    cur = conn.cursor()
    s3 = get_s3()

    # Fetch the media_key to verify the file exists
    cur.execute("SELECT media_key FROM posts WHERE id = %s", (post_id,))
    row = cur.fetchone()
    if not row:
        conn.close()
        return {"error": "Post not found"}

    media_key = row[0]

    # Verify the file actually exists in MinIO
    try:
        s3.head_object(Bucket=BUCKET_NAME, Key=media_key)
    except Exception:
        conn.close()
        return {"error": "File not found in storage"}

    # Mark as complete
    cur.execute(
        "UPDATE posts SET media_upload_status = 'complete' WHERE id = %s",
        (post_id,)
    )
    conn.commit()
    conn.close()
    return {"post_id": post_id, "status": "complete"}

confirmation = confirm_upload(result["post_id"])
assert confirmation.get("status") == "complete", confirmation
print(f"✅ Post #{confirmation['post_id']} is now {confirmation['status']}")
print(f"   This post will now appear in followers' feeds!")

# ── The rejection path has to actually reject, or it is decoration ──────
# Create a post whose client never uploaded anything, then confirm it.
ghost = create_post(author_id=1, caption="Client crashed before uploading")
ghost_result = confirm_upload(ghost["post_id"])
assert ghost_result.get("error") == "File not found in storage", ghost_result

conn = get_db()
cur = conn.cursor()
cur.execute("SELECT media_upload_status FROM posts WHERE id = %s", (ghost["post_id"],))
ghost_status = cur.fetchone()[0]
conn.close()
assert ghost_status == "pending", f"expected 'pending', got {ghost_status!r}"

print(f"\n🚫 Post #{ghost['post_id']} rejected: {ghost_result['error']}")
print(f"   Its status stays '{ghost_status}' — Notebook 2's feed queries filter")
print(f"   on media_upload_status, so a half-uploaded post never reaches a feed.")

## 📏 Generating Thumbnails (Multiple Image Sizes)

Instagram doesn't serve the same image to everyone. Consider:
- A user on a **phone** with a small screen doesn't need a 4000×3000 photo
- A user on a **slow 3G connection** can't wait for a 5MB image
- A **thumbnail** in the profile grid is only 150×150 pixels

So Instagram generates **multiple sizes** of every uploaded photo:

| Size | Dimensions | Use Case |
|------|-----------|----------|
| `thumbnail` | 150×150 | Profile grid, search results |
| `small` | 320×320 | Stories preview, notifications |
| `medium` | 640×640 | Feed on mobile devices |
| `large` | 1080×1080 | Feed on tablets/desktops |
| `original` | As uploaded | Zoom view, download |

This happens **asynchronously** after the upload — the user doesn't wait for it.

Two details that are easy to get wrong:

- Those dimensions are **square**, so the variant is a centre *crop* plus a resize.
  Resizing alone only preserves the aspect ratio — an 800×600 upload would give
  you a 150×112 "150×150 thumbnail".
- Variants are **never upscaled**. There is no honest 1080×1080 version of an
  800×600 upload, so the largest variant is capped at the original's short side.

In [ ]:
from PIL import ImageOps

THUMBNAIL_SIZES = {
    "thumbnail": (150, 150),
    "small": (320, 320),
    "medium": (640, 640),
    "large": (1080, 1080),
}

def generate_thumbnails(media_key: str):
    """
    Downloads the original image from MinIO, creates resized versions,
    and uploads each variant back to MinIO.

    `ImageOps.fit` centre-crops *and* resizes in one step, which is what we
    want: Instagram's variants are square. `Image.thumbnail` on its own only
    scales, so it would happily call a 150×112 image a "150×150 thumbnail".

    In production, this would be a background job (e.g., triggered by
    an S3 event notification → Lambda function or a message queue worker).
    """
    s3 = get_s3()

    # Download the original image
    response = s3.get_object(Bucket=BUCKET_NAME, Key=media_key)
    original_bytes = response["Body"].read()
    original_image = Image.open(io.BytesIO(original_bytes))

    # The biggest square we can cut out of this photo without inventing pixels.
    short_side = min(original_image.size)

    print(f"📷 Original image: {original_image.size[0]}×{original_image.size[1]}")
    print(f"   Size: {len(original_bytes):,} bytes")
    print(f"   Largest honest square crop: {short_side}×{short_side}")
    print()

    results = []
    for size_name, (want_w, want_h) in THUMBNAIL_SIZES.items():
        # Never upscale — a variant bigger than the source is pure waste.
        edge = min(want_w, want_h, short_side)
        capped = edge < want_w

        img_copy = ImageOps.fit(
            original_image, (edge, edge), method=Image.Resampling.LANCZOS
        )

        # Save to bytes
        buffer = io.BytesIO()
        img_copy.save(buffer, format="JPEG", quality=85)
        buffer.seek(0)

        # Upload variant to MinIO — note the key pattern!
        # Original: photos/user_1/post_42.jpg
        # Variant:  photos/user_1/post_42_thumbnail.jpg
        variant_key = media_key.replace(".jpg", f"_{size_name}.jpg")
        s3.put_object(
            Bucket=BUCKET_NAME,
            Key=variant_key,
            Body=buffer.getvalue(),
            ContentType="image/jpeg"
        )

        saved_size = len(buffer.getvalue())
        results.append((size_name, img_copy.size, saved_size))
        note = f"  (capped: source is only {short_side}px tall/wide)" if capped else ""
        print(f"   ✅ {size_name:12s} → {img_copy.size[0]:4d}×{img_copy.size[1]:<4d} "
              f"({saved_size:>6,} bytes){note}")

    return results

print("Generating thumbnails...\n")
thumbnails = generate_thumbnails(result["media_key"])

# ── The table above promises square variants that get cheaper as they shrink.
#    Assert it, so a future refactor cannot quietly go back to 150×112.
sizes = [sz for _, sz, _ in thumbnails]
assert all(w == h for w, h in sizes), f"variants must be square, got {sizes}"
edges = [w for w, _ in sizes]
assert edges == sorted(edges), f"variants must not shrink as the name grows: {edges}"
byte_sizes = [b for _, _, b in thumbnails]
assert byte_sizes[0] < byte_sizes[-1], (
    f"the thumbnail must be cheaper to serve than the largest variant: {byte_sizes}")

saving = 1 - byte_sizes[0] / byte_sizes[-1]
print(f"\n🎉 Generated {len(thumbnails)} variants + original")
print(f"   Grid thumbnail is {saving:.0%} smaller than the largest variant")
print(f"   → Open MinIO Console (http://localhost:9001) to see all variants")

## 🌐 How a CDN Serves Images

In production, users never download images directly from S3.  
Instead, a **CDN (Content Delivery Network)** like CloudFront sits in front of S3.

```
User in Tokyo                         User in New York
     │                                      │
     ▼                                      ▼
┌─────────────┐                    ┌─────────────┐
│ CDN Edge    │                    │ CDN Edge    │
│ Tokyo       │                    │ New York    │
│ (cached!)   │                    │ (cached!)   │
└──────┬──────┘                    └──────┬──────┘
       │ cache miss (first time)          │
       ▼                                  ▼
┌──────────────────────────────────────────────┐
│              S3 Origin (us-east-1)           │
│  photos/user_7/post_42.jpg                   │
└──────────────────────────────────────────────┘
```

**How it works:**
1. User requests `https://cdn.instagram.com/photos/user_7/post_42_medium.jpg`
2. CDN edge in Tokyo checks: "Do I have this cached?"  
   - **Yes** → return instantly (< 50ms)  
   - **No** → fetch from S3 origin, cache it, return (~200ms first time)
3. Next user in Tokyo requesting the same image gets it from cache

We can't demo a real CDN locally, but let's simulate the **read path**:

In [ ]:
import redis

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": False  # We'll store binary image data
}

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

def get_image(media_key: str, size: str = "medium") -> bytes:
    """
    Simulates how a CDN serves images:
    1. Check Redis cache (simulates CDN edge cache)
    2. If miss → fetch from MinIO (simulates S3 origin)
    3. Cache the result for next time
    """
    r = get_redis()
    s3 = get_s3()

    variant_key = media_key.replace(".jpg", f"_{size}.jpg")
    cache_key = f"img:{variant_key}"

    # 1. Check cache
    start = time.time()
    cached = r.get(cache_key)
    if cached:
        elapsed = (time.time() - start) * 1000
        print(f"⚡ Cache HIT — {len(cached):,} bytes in {elapsed:.1f}ms")
        return cached

    # 2. Cache miss — fetch from MinIO (origin)
    response = s3.get_object(Bucket=BUCKET_NAME, Key=variant_key)
    image_bytes = response["Body"].read()

    # 3. Cache it (TTL = 1 hour, simulating CDN edge cache)
    r.setex(cache_key, 3600, image_bytes)
    elapsed = (time.time() - start) * 1000
    print(f"🔄 Cache MISS — fetched from origin, {len(image_bytes):,} bytes in {elapsed:.1f}ms")
    return image_bytes

# Start from a cold edge so the first request is genuinely a miss.
medium_key = result["media_key"].replace(".jpg", "_medium.jpg")
cache_key = f"img:{medium_key}"
get_redis().delete(cache_key)

print("First request (cache miss → fetches from MinIO):")
img1 = get_image(result["media_key"], "medium")

print("\nSecond request (cache hit → served from Redis):")
img2 = get_image(result["media_key"], "medium")

# A cache that serves different bytes than the origin is worse than no cache.
assert get_redis().exists(cache_key), "the second read should have been a cache hit"
assert img1 == img2, "cached bytes differ from the origin bytes"
assert 0 < get_redis().ttl(cache_key) <= 3600, "edge entries must expire, not live forever"

print("\n💡 In production, the CDN cache hit rate is 90%+")
print("   Most images are served without ever touching S3!")

## 📊 The Complete Upload Pipeline

Let's put it all together and see the full flow:

```
┌────────┐    ①POST /posts      ┌──────────┐    ②INSERT        ┌────────────┐
│ Client │──────────────────────►│   App    │─────────────────►│ PostgreSQL │
│        │◄──────────────────────│  Server  │                  │            │
│        │  {postId, presignUrl} │          │                  │ media_key  │
│        │                       └──────────┘                  │ status=    │
│        │                                                     │  pending   │
│        │    ③PUT presignedUrl   ┌──────────┐                └────────────┘
│        │───────────────────────►│  MinIO   │
│        │   (photo bytes)        │  (S3)    │
│        │                        └──────────┘
│        │    ④PATCH /posts/{id}  ┌──────────┐   ⑤UPDATE       ┌────────────┐
│        │──────────────────────►│   App    │────────────────►│ PostgreSQL │
│        │                       │  Server  │                 │ status=    │
└────────┘                       └────┬─────┘                 │  complete  │
                                      │                       └────────────┘
                                      │ ⑥ async job
                                      ▼
                               ┌──────────┐
                               │Thumbnail │──► MinIO (variants)
                               │ Worker   │──► Fan-out to feeds
                               └──────────┘
```

In [ ]:
def full_upload_pipeline(author_id: int, caption: str, image_color: str = "skyblue"):
    """
    Runs the complete Instagram upload pipeline end-to-end.
    """
    print("=" * 60)
    print(f"📸 UPLOAD PIPELINE for user {author_id}")
    print("=" * 60)

    # Step 1: Create post metadata + get pre-signed URL
    print("\n① Creating post metadata...")
    post = create_post(author_id, caption)
    print(f"   Post #{post['post_id']} created (status: pending)")

    # Step 2: Upload photo to MinIO via pre-signed URL
    print("\n② Uploading photo to MinIO...")
    photo = create_test_image(1080, 1080, image_color)
    resp = requests.put(
        post["presigned_url"],
        data=photo.getvalue(),
        headers={"Content-Type": "image/jpeg"}
    )
    print(f"   Uploaded {len(photo.getvalue()):,} bytes → {resp.status_code}")

    # Step 3: Confirm upload
    print("\n③ Confirming upload...")
    confirm = confirm_upload(post["post_id"])
    print(f"   Status: {confirm['status']}")

    # Step 4: Generate thumbnails (async in production)
    print("\n④ Generating thumbnails...")
    thumbs = generate_thumbnails(post["media_key"])

    print("\n" + "=" * 60)
    print(f"✅ Post #{post['post_id']} is live!")
    print("=" * 60)
    return post

# Run the full pipeline!
new_post = full_upload_pipeline(
    author_id=5,
    caption="Coffee and code ☕💻",
    image_color="brown"
)

## 🧠 Key Takeaways

1. **Separate metadata from media** — database stores post info, object storage (S3) stores photos
2. **Pre-signed URLs** — let clients upload directly to S3, keeping your app server lightweight
3. **Three-step upload** — create metadata → upload media → confirm completion
4. **Generate multiple sizes** — serve the right image for each device/network condition
5. **CDN caching** — cache images at edge locations worldwide for < 50ms delivery

### Interview Tips

- Always mention pre-signed URLs when discussing large file uploads
- Explain that thumbnail generation happens **asynchronously** (via a queue/Lambda)
- Mention that in production, S3 event notifications can trigger the thumbnail worker
- Talk about CDN cache hit rates (typically 90%+) to show you understand the scale

### What's Next?

In **Notebook 2**, we'll see how this uploaded post gets pushed into followers' feeds  
using fan-out on write — the core of Instagram's feed generation system.